# Benchmark bioplastics metadata versus no bioplastics metadata

This notebook measures the database overhead and analytical capabilities of the corrected bioplastics/biochemical metadata layer. Shared sequence queries run 30 measured repetitions after five warmups, with result hashes and query plans persisted. Metadata-only capabilities are timed separately and never assigned fabricated values in the unannotated database.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))
from bioplastics_benchmark_core import (
    benchmark_bioplastics_databases,
    benchmark_bioplastics_capabilities,
    ROOT,
    bioplastics_storage_summary,
)
root = ROOT

In [2]:
raw, query_summary, proofs, plans = benchmark_bioplastics_databases(root)
display(query_summary)
assert all(item['equivalent'] for item in proofs.values())
print(f'Equivalent shared-query result hashes: {len(proofs)} of {len(proofs)}')
print(f'Saved query plans: {sum(len(value) for value in plans.values())}')

,variant,query,median_ms,mean_ms,stddev_ms,minimum_ms,maximum_ms,p95_ms,workloads_per_second
0,bioplastics_metadata,average_sequence_length,16.286875,16.376481,0.472877,15.384072,17.912505,17.683438,61.399133
1,bioplastics_metadata,count_all_sequences,0.033407,0.035694,0.008903,0.032617,0.078702,0.053131,29933.398189
2,bioplastics_metadata,gc_range_count,5.017680,5.248135,0.674498,4.971556,8.337094,6.630216,199.295292
3,bioplastics_metadata,plastic_context_sequence_count,1.064253,1.124413,0.384511,0.996206,3.152022,1.121979,939.625756
4,bioplastics_metadata,top_100_longest,0.097004,0.099987,0.008934,0.094076,0.134067,0.122647,10308.853243
5,no_bioplastics_metadata,average_sequence_length,16.125053,16.430774,0.849168,15.496440,19.334972,18.593616,62.015300
6,no_bioplastics_metadata,count_all_sequences,0.033997,0.164435,0.568899,0.033042,3.101430,0.726623,29413.927495
7,no_bioplastics_metadata,gc_range_count,5.154047,5.210909,0.217808,4.880407,5.907286,5.740837,194.022290
8,no_bioplastics_metadata,plastic_context_sequence_count,1.078793,1.101787,0.053264,1.058956,1.255380,1.231028,926.962321
9,no_bioplastics_metadata,top_100_longest,0.100385,0.125213,0.058977,0.098423,0.327505,0.270115,9961.647657


Equivalent shared-query result hashes: 5 of 5
Saved query plans: 10


In [3]:
storage = bioplastics_storage_summary(root)
display(storage)
ratio = storage.set_index('variant').loc['bioplastics_metadata','database_mib'] / storage.set_index('variant').loc['no_bioplastics_metadata','database_mib']
print(f'Bioplastics-metadata storage ratio: {ratio:.3f}x')

,variant,database_mib,page_count,page_size,integrity_check,sequence_rows,reference_rows,annotation_rows,homology_hit_rows
0,no_bioplastics_metadata,39.691406,10161,4096,ok,121326,0,0,0
1,bioplastics_metadata,40.050781,10253,4096,ok,121326,323,534,28


Bioplastics-metadata storage ratio: 1.009x


## Metadata-enabled analyses

These queries directly exercise the study metadata: permissive similarity candidates grouped by reference plastic and enzyme family, matches to references in the broad bioplastic-or-biodegradable polymer class, separation of study context from molecular similarity, reference biochemistry by plastic class, and reference coverage by PlasticDB versus PAZy. Commercial formulations and natural biopolymers are separate classes.

In [4]:
capabilities, capability_plans = benchmark_bioplastics_capabilities(root)
display(capabilities[['capability','median_ms','p95_ms','repetitions','result_rows','result']])
print(f'Saved metadata-capability plans: {len(capability_plans)}')

,capability,median_ms,p95_ms,repetitions,result_rows,result
0,candidate_sequences_by_reference_plastic,0.233262,0.282977,30,11,"[[""PEG"", 6], [""PHA"", 6], [""PCL"", 4], [""PHBV"", ..."
1,candidate_sequences_by_reference_enzyme_family,0.191130,0.231129,30,4,"[[""Other enzyme"", 12], [""Depolymerase"", 4], [""..."
2,candidate_sequences_matching_bioplastic_or_bio...,0.115031,0.152932,30,1,[[12]]
3,hits_by_study_context,0.198627,0.293850,30,1,"[[""background_control"", 18]]"
4,reference_biochemistry_by_plastic_class,0.825519,0.904629,30,4,"[[""bioplastic_or_biodegradable_polymer"", 132, ..."
5,reference_coverage_by_source,0.502879,0.539768,30,2,"[[""PAZy"", 12, 13], [""PlasticDB"", 313, 521]]"


Saved metadata-capability plans: 6


## Interpretation limits

- The benchmark measures local SQLite behavior on this reproducible sample, not distributed Logan Search or AWS Batch performance.
- Study titles establish plastic-related experimental context, not degradation capability.
- Candidate homology rows are a permissive similarity screen against curated plastic-active proteins; the thresholds are not calibrated to infer biochemical function.
- PAZy accessions that could not be resolved are reported, not silently substituted.
- Storage overhead includes the complete reference, annotation, biochemical, homology, context, index, and provenance layer.